## Silver specs (in order)

| | Step | Rule |
|---|---|---|
| ✅ | Key conflicts | 13 keys have 2 rows with different values → keep 1 row per key |
| ✅ | Full duplicates | 380,454 exact copies → delete (same step: `dropDuplicates` on the 5 key columns) |
| ✅  | event_time | text → timestamp |
| ✅ | event_date (new) | date from event_time → for daily and monthly counts |
| ⬜ | price | text → decimal(10,2) · 0 → NULL |
| ⬜ | brand | NULL → 'unknown' |
| ⬜ | category_code | NULL → 'unknown' |
| ⬜ | user_session | NULL → stays NULL (nothing to do) |
| ⬜ | Save | `dev.ecommerce.events_silver` |
| ⬜ | Check | 233,080,195 rows · key unique · types correct |

## Decisions in this notebook
- **13 key conflicts → keep one row per key.** They are real cart events, only one column value is unclear, and 13 rows out of 233M don't change any result. 11 of them are on 16-17 Nov, during the known cart tracking problem.

**Expected silver rows:** 233,460,662 − 380,454 − 13 = **233,080,195** ✅ (matched after dedup)

In [0]:
%sql
use catalog dev;
use schema ecommerce;

In [0]:
%sql
select * from events_bronze

In [0]:
%sql
-- Rows with the same key but different values in other columns.
-- Step 1: remove exact duplicates (all 9 data columns the same)
WITH unique_rows AS (
    SELECT DISTINCT
        event_time, event_type, product_id, user_id, user_session,
        category_id, category_code, brand, price
    FROM events_bronze
)
-- Step 2: keys that still have more than one row
SELECT
    event_time, event_type, product_id, user_id, user_session,
    COUNT(*) AS rows_per_key
FROM unique_rows
GROUP BY event_time, event_type, product_id, user_id, user_session
HAVING COUNT(*) > 1
ORDER BY event_time



In [0]:
bronze_df = spark.read.table("dev.ecommerce.events_bronze")

In [0]:
bronze_df.count()

In [0]:
silver_df = bronze_df.dropDuplicates(["event_time", "event_type","product_id", "user_id", "user_session"])

silver_df.count()

In [0]:
silver_df.distinct().count()

In [0]:
from pyspark.sql.functions import col, to_timestamp, to_date

silver_df = silver_df.withColumns({
    "event_time" : to_timestamp(col("event_time"), 'yyyy-MM-dd HH:mm:ss z')
})

silver_df.printSchema()

In [0]:
silver_df.filter(col("event_time").isNull()).count()

In [0]:
silver_df = silver_df.withColumns({
    "event_date" : to_date(col("event_time"))
})

silver_df.select("event_date").display()